In [1]:
!pip install gdown -q

In [2]:
!gdown --id 1sHYkAuE-5Z0A1NBFxblvlQ8o8bZZuDyl

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1sHYkAuE-5Z0A1NBFxblvlQ8o8bZZuDyl
From (redirected): https://drive.google.com/uc?id=1sHYkAuE-5Z0A1NBFxblvlQ8o8bZZuDyl&confirm=t&uuid=204a7a96-0089-4620-855e-e7c40ab9fb8b
To: /content/Preprocesado_Pix2Pix.zip
100% 1.04G/1.04G [00:14<00:00, 70.7MB/s]


In [3]:
!unzip -q Preprocesado_Pix2Pix.zip

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import time
from tqdm import tqdm

print(f"TensorFlow Version: {tf.__version__}")

TensorFlow Version: 2.19.0


In [7]:
# --- 1. Definición de Parámetros ---
print("Paso 1: Definiendo parámetros...")

# Ruta al dataset preprocesado
PATH = 'Preprocesado_Pix2Pix/'
TRAIN_A_DIR = os.path.join(PATH, 'train_A')
TRAIN_B_DIR = os.path.join(PATH, 'train_B')

# Directorio de salida para las imágenes de prueba
OUTPUT_IMG_DIR = 'Resultados_Entrenamiento'
os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)

# Parámetros de las imágenes
BUFFER_SIZE = 400  # Un valor más bajo si tienes poca RAM, pero 400 es bueno
BATCH_SIZE = 1     # Pix2Pix funciona mejor con BATCH_SIZE = 1
IMG_WIDTH = 256
IMG_HEIGHT = 256

# Parámetros de entrenamiento
EPOCHS = 10        # Básico/Preliminar. Un entrenamiento real necesita 100-200.
LAMBDA = 100       # Ponderación de la pérdida L1 (estándar en Pix2Pix)


Paso 1: Definiendo parámetros...


In [9]:
# --- 2. Funciones de Carga y Preprocesamiento de Datos ---
print("Paso 2: Definiendo funciones de carga de datos...")

def load_image(image_file):
    """Carga una imagen y la divide en T0 (entrada) y T1 (real)."""
    # image_file es la RUTA a la imagen en train_A
    # Necesitamos encontrar su par correspondiente en train_B

    # Construimos la ruta al par en train_B
    # Reemplazamos la ruta base 'train_A' por 'train_B'
    real_image_file = tf.strings.regex_replace(image_file, "train_A", "train_B")

    # Leemos y decodificamos ambas imágenes
    input_image = tf.io.read_file(image_file)
    input_image = tf.io.decode_jpeg(input_image, channels=3)

    real_image = tf.io.read_file(real_image_file)
    real_image = tf.io.decode_jpeg(real_image, channels=3)

    # Convertir a float32
    input_image = tf.cast(input_image, tf.float32)
    real_image = tf.cast(real_image, tf.float32)

    return input_image, real_image

Paso 2: Definiendo funciones de carga de datos...


In [10]:
def resize(input_image, real_image, height, width):
    """Redimensiona ambas imágenes."""
    input_image = tf.image.resize(input_image, [height, width],
                                  method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
    real_image = tf.image.resize(real_image, [height, width],
                                 method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
    return input_image, real_image

In [12]:
def random_crop(input_image, real_image):
    """Recorte aleatorio a 256x256 (aumentación de datos)."""
    stacked_image = tf.stack([input_image, real_image], axis=0)
    cropped_image = tf.image.random_crop(
        stacked_image, size=[2, IMG_HEIGHT, IMG_WIDTH, 3])
    return cropped_image[0], cropped_image[1]

def normalize(input_image, real_image):
    """Normaliza las imágenes al rango [-1, 1]."""
    input_image = (input_image / 127.5) - 1
    real_image = (real_image / 127.5) - 1
    return input_image, real_image

@tf.function()
def random_jitter(input_image, real_image):
    """Aumentación de datos estándar de Pix2Pix."""
    # 1. Redimensionar a 286x286
    input_image, real_image = resize(input_image, real_image, 286, 286)

    # 2. Recortar aleatoriamente de nuevo a 256x256
    input_image, real_image = random_crop(input_image, real_image)

    # 3. Volteo horizontal aleatorio
    if tf.random.uniform(()) > 0.5:
        input_image = tf.image.flip_left_right(input_image)
        real_image = tf.image.flip_left_right(real_image)

    return input_image, real_image

def load_image_train(image_file):
    """Carga, aplica jitter y normaliza la imagen de entrenamiento."""
    input_image, real_image = load_image(image_file)
    input_image, real_image = random_jitter(input_image, real_image)
    input_image, real_image = normalize(input_image, real_image)
    return input_image, real_image
def load_image_test(image_file):
    """Carga y normaliza una imagen de prueba (sin jitter)."""
    input_image, real_image = load_image(image_file)
    input_image, real_image = resize(input_image, real_image, IMG_HEIGHT, IMG_WIDTH)
    input_image, real_image = normalize(input_image, real_image)
    return input_image, real_image

In [13]:
# --- 3. Crear el Pipeline de Datos (tf.data) ---
print("Paso 3: Creando pipeline de tf.data...")

# Obtener la lista de archivos de entrada (T0)
train_files_pattern = os.path.join(TRAIN_A_DIR, '*.jpg')
num_train_files = len(glob.glob(train_files_pattern))

if num_train_files == 0:
    print(f"¡ADVERTENCIA! No se encontraron imágenes en: {train_files_pattern}")
    print("Asegúrate de haber ejecutado el script de preprocesamiento anterior.")
else:
    print(f"Total de pares de entrenamiento encontrados: {num_train_files}")

train_dataset = tf.data.Dataset.list_files(train_files_pattern)
train_dataset = train_dataset.map(load_image_train,
                                  num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(BUFFER_SIZE)
train_dataset = train_dataset.batch(BATCH_SIZE)
train_dataset = train_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

# Usaremos un pequeño subconjunto (10 imágenes) del mismo set para visualización
test_dataset = tf.data.Dataset.list_files(train_files_pattern)
test_dataset = test_dataset.take(10)
test_dataset = test_dataset.map(load_image_test)
test_dataset = test_dataset.batch(BATCH_SIZE)
test_dataset = test_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

Paso 3: Creando pipeline de tf.data...
Total de pares de entrenamiento encontrados: 1956


In [16]:
# --- 4. Construir el Generador (U-Net) ---
print("Paso 4: Construyendo el Generador (U-Net)...")

def downsample(filters, size, apply_batchnorm=True):
    """Bloque de codificación (Downsampling) en U-Net."""
    initializer = tf.random_normal_initializer(0., 0.02)
    result = tf.keras.Sequential()
    result.add(
        layers.Conv2D(filters, size, strides=2, padding='same',
                             kernel_initializer=initializer, use_bias=False))
    if apply_batchnorm:
        result.add(layers.BatchNormalization())
    result.add(layers.LeakyReLU())
    return result

def upsample(filters, size, apply_dropout=False):
    """Bloque de decodificación (Upsampling) en U-Net."""
    initializer = tf.random_normal_initializer(0., 0.02)
    result = tf.keras.Sequential()
    result.add(
        layers.Conv2DTranspose(filters, size, strides=2,
                                      padding='same',
                                      kernel_initializer=initializer,
                                      use_bias=False))
    result.add(layers.BatchNormalization())
    if apply_dropout:
        result.add(layers.Dropout(0.5))
    result.add(layers.ReLU())
    return result

def build_generator():
    """Construye el Generador U-Net completo."""
    inputs = layers.Input(shape=[IMG_WIDTH, IMG_HEIGHT, 3])

    # Encoder
    down_stack = [
        downsample(64, 4, apply_batchnorm=False),  # (bs, 128, 128, 64)
        downsample(128, 4),  # (bs, 64, 64, 128)
        downsample(256, 4),  # (bs, 32, 32, 256)
        downsample(512, 4),  # (bs, 16, 16, 512)
        downsample(512, 4),  # (bs, 8, 8, 512)
        downsample(512, 4),  # (bs, 4, 4, 512)
        downsample(512, 4),  # (bs, 2, 2, 512)
        downsample(512, 4),  # (bs, 1, 1, 512)
    ]

    # Decoder
    up_stack = [
        upsample(512, 4, apply_dropout=True),  # (bs, 2, 2, 1024)
        upsample(512, 4, apply_dropout=True),  # (bs, 4, 4, 1024)
        upsample(512, 4, apply_dropout=True),  # (bs, 8, 8, 1024)
        upsample(512, 4),  # (bs, 16, 16, 1024)
        upsample(256, 4),  # (bs, 32, 32, 512)
        upsample(128, 4),  # (bs, 64, 64, 256)
        upsample(64, 4),   # (bs, 128, 128, 128)
    ]

    initializer = tf.random_normal_initializer(0., 0.02)
    last = layers.Conv2DTranspose(3, 4,  # 3 canales de salida (RGB)
                                  strides=2,
                                  padding='same',
                                  kernel_initializer=initializer,
                                  activation='tanh')  # Salida en rango [-1, 1]

    x = inputs

    # Conectar el Encoder
    skips = []
    for down in down_stack:
        x = down(x)
        skips.append(x)

    skips = reversed(skips[:-1])

    # Conectar el Decoder con las Skip-Connections
    for up, skip in zip(up_stack, skips):
        x = up(x)
        x = layers.Concatenate()([x, skip])

    x = last(x) # (bs, 256, 256, 3)

    return Model(inputs=inputs, outputs=x)

generator = build_generator()
#tf.keras.utils.plot_model(generator, show_shapes=True, dpi=64)

Paso 4: Construyendo el Generador (U-Net)...


In [18]:
# --- 5. Construir el Discriminador (PatchGAN) ---
print("Paso 5: Construyendo el Discriminador (PatchGAN)...")

def build_discriminator():
    initializer = tf.random_normal_initializer(0., 0.02)

    # El discriminador recibe la imagen de entrada (T0)
    inp = layers.Input(shape=[IMG_WIDTH, IMG_HEIGHT, 3], name='input_image')
    # Y la imagen objetivo (T1) o generada (T_pred)
    tar = layers.Input(shape=[IMG_WIDTH, IMG_HEIGHT, 3], name='target_image')

    # Concatenamos ambas imágenes en el eje de canales (3 + 3 = 6 canales)
    x = layers.concatenate([inp, tar])  # (bs, 256, 256, 6)

    down1 = downsample(64, 4, False)(x)  # (bs, 128, 128, 64)
    down2 = downsample(128, 4)(down1)     # (bs, 64, 64, 128)
    down3 = downsample(256, 4)(down2)     # (bs, 32, 32, 256)

    # Capa intermedia (padding + conv)
    zero_pad1 = layers.ZeroPadding2D()(down3)  # (bs, 34, 34, 256)
    conv = layers.Conv2D(512, 4, strides=1,
                          kernel_initializer=initializer,
                          use_bias=False)(zero_pad1)  # (bs, 31, 31, 512)

    batchnorm1 = layers.BatchNormalization()(conv)
    leaky_relu = layers.LeakyReLU()(batchnorm1)

    # Capa final
    zero_pad2 = layers.ZeroPadding2D()(leaky_relu)  # (bs, 33, 33, 512)

    # Salida de 1 canal para la predicción Real/Falso (Patch)
    last = layers.Conv2D(1, 4, strides=1,
                          kernel_initializer=initializer)(zero_pad2)  # (bs, 30, 30, 1)

    # No se usa 'sigmoid' aquí, la pérdida 'from_logits=True' es más estable
    return Model(inputs=[inp, tar], outputs=last)

discriminator = build_discriminator()
#tf.keras.utils.plot_model(discriminator, show_shapes=True, dpi=64)

Paso 5: Construyendo el Discriminador (PatchGAN)...


In [19]:
# --- 6. Definición de Funciones de Pérdida y Optimizadores ---
print("Paso 6: Definiendo pérdidas y optimizadores...")

loss_object = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(disc_real_output, disc_generated_output):
    """Pérdida del Discriminador."""
    # Pérdida para imágenes reales (queremos que sean '1')
    real_loss = loss_object(tf.ones_like(disc_real_output), disc_real_output)

    # Pérdida para imágenes generadas (queremos que sean '0')
    generated_loss = loss_object(tf.zeros_like(disc_generated_output), disc_generated_output)

    total_disc_loss = real_loss + generated_loss
    return total_disc_loss

def generator_loss(disc_generated_output, gen_output, target):
    """Pérdida del Generador."""
    # Pérdida GAN: ¿Qué tan bien engañamos al discriminador? (queremos que sean '1')
    gan_loss = loss_object(tf.ones_like(disc_generated_output), disc_generated_output)

    # Pérdida de Reconstrucción (L1 / MAE)
    l1_loss = tf.reduce_mean(tf.abs(target - gen_output))

    # Pérdida total = Pérdida GAN + (LAMBDA * Pérdida L1)
    total_gen_loss = gan_loss + (LAMBDA * l1_loss)

    return total_gen_loss, gan_loss, l1_loss

# Optimizadores (como se especifica en el paper de Pix2Pix)
generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

# Checkpoints (para guardar el progreso)
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)

Paso 6: Definiendo pérdidas y optimizadores...


In [20]:
# --- 7. Función de Generación de Imágenes (Visualización) ---
print("Paso 7: Definiendo función de visualización...")

def generate_images(model, test_input, tar, epoch):
    """Genera y guarda una imagen de muestra."""
    prediction = model(test_input, training=True)

    plt.figure(figsize=(15, 5))

    display_list = [test_input[0], tar[0], prediction[0]]
    title = ['Input Image (T0)', 'Ground Truth (T1)', 'Predicted Image (T_pred)']

    for i in range(3):
        plt.subplot(1, 3, i+1)
        plt.title(title[i])
        # Desnormalizar la imagen de [-1, 1] a [0, 1] para plt
        plt.imshow(display_list[i] * 0.5 + 0.5)
        plt.axis('off')

    # Guardar la figura
    save_path = os.path.join(OUTPUT_IMG_DIR, f'image_at_epoch_{epoch:04d}.png')
    plt.savefig(save_path)
    plt.close()

Paso 7: Definiendo función de visualización...


In [21]:
# --- 8. Paso de Entrenamiento (Train Step) ---
print("Paso 8: Definiendo el paso de entrenamiento...")

@tf.function
def train_step(input_image, target, epoch):
    """Ejecuta un solo paso de entrenamiento."""
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        # 1. Generar imagen (T_pred)
        gen_output = generator(input_image, training=True)

        # 2. Correr el Discriminador
        # Par Real: (T0, T1)
        disc_real_output = discriminator([input_image, target], training=True)
        # Par Falso: (T0, T_pred)
        disc_generated_output = discriminator([input_image, gen_output], training=True)

        # 3. Calcular Pérdidas
        gen_total_loss, gen_gan_loss, gen_l1_loss = generator_loss(disc_generated_output, gen_output, target)
        disc_loss = discriminator_loss(disc_real_output, disc_generated_output)

    # 4. Calcular Gradientes
    generator_gradients = gen_tape.gradient(gen_total_loss,
                                            generator.trainable_variables)
    discriminator_gradients = disc_tape.gradient(disc_loss,
                                                 discriminator.trainable_variables)

    # 5. Aplicar Gradientes
    generator_optimizer.apply_gradients(zip(generator_gradients,
                                            generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(discriminator_gradients,
                                                discriminator.trainable_variables))

    return disc_loss, gen_total_loss, gen_gan_loss, gen_l1_loss

Paso 8: Definiendo el paso de entrenamiento...


In [22]:
# --- 9. Bucle de Entrenamiento ---
print(f"Paso 9: Iniciando el bucle de entrenamiento por {EPOCHS} epochs...")
print(f"Los resultados se guardarán en: {OUTPUT_IMG_DIR}")

def train(dataset, epochs):
    if num_train_files == 0:
        print("¡ERROR! No se encontraron imágenes en el directorio de entrenamiento.")
        print("Asegúrate de que el script de preprocesamiento se haya ejecutado correctamente.")
        return

    # Preparar el lote de prueba para visualización (solo una vez)
    try:
        test_batch = next(iter(test_dataset))
    except StopIteration:
        print("¡ERROR! El dataset de prueba está vacío. No se puede generar imagen de muestra.")
        return

    for epoch in range(epochs):
        start_epoch = time.time()

        # Generar y guardar una imagen de muestra al inicio de la época
        print(f"\nGenerando imagen de muestra para la época {epoch + 1}...")
        generate_images(generator, test_batch[0], test_batch[1], epoch)

        print(f"Iniciando Época {epoch + 1}/{epochs}")

        # Inicializar barra de progreso (tqdm)
        pbar = tqdm(total=num_train_files // BATCH_SIZE, desc=f"Época {epoch + 1}", unit="batch")

        epoch_disc_loss = []
        epoch_gen_total_loss = []
        epoch_gen_gan_loss = []
        epoch_gen_l1_loss = []

        for n, (input_image, target) in dataset.enumerate():
            # Ejecutar el paso de entrenamiento
            disc_loss, gen_total_loss, gen_gan_loss, gen_l1_loss = train_step(input_image, target, epoch)

            # Almacenar pérdidas
            epoch_disc_loss.append(disc_loss.numpy())
            epoch_gen_total_loss.append(gen_total_loss.numpy())
            epoch_gen_gan_loss.append(gen_gan_loss.numpy())
            epoch_gen_l1_loss.append(gen_l1_loss.numpy())

            # Actualizar barra de progreso
            pbar.update(1)
            if (n + 1) % 50 == 0:
                pbar.set_postfix_str(f"D_loss: {np.mean(epoch_disc_loss):.4f}, G_loss_total: {np.mean(epoch_gen_total_loss):.4f}")

        pbar.close()

        # Imprimir resumen de la época
        print(f"\n--- Resumen Época {epoch + 1} ---")
        print(f"Tiempo: {time.time() - start_epoch:.2f} seg")
        print(f"Pérdida Discriminador (Media): {np.mean(epoch_disc_loss):.4f}")
        print(f"Pérdida Generador (Total):     {np.mean(epoch_gen_total_loss):.4f}")
        print(f"Pérdida Generador (GAN):       {np.mean(epoch_gen_gan_loss):.4f}")
        print(f"Pérdida Generador (L1):        {np.mean(epoch_gen_l1_loss):.4f}")
        print("---------------------------\n")

        # Guardar checkpoint
        if (epoch + 1) % 5 == 0:
            checkpoint.save(file_prefix=checkpoint_prefix)
            print(f"Checkpoint guardado para la época {epoch + 1} en {checkpoint_dir}")

    # Generar una imagen final al terminar
    print("Generando imagen final del entrenamiento...")
    generate_images(generator, test_batch[0], test_batch[1], epochs)

Paso 9: Iniciando el bucle de entrenamiento por 10 epochs...
Los resultados se guardarán en: Resultados_Entrenamiento


In [23]:
# --- Iniciar el entrenamiento ---
train(train_dataset, EPOCHS)

print("¡Entrenamiento completado!")
print(f"Puedes ver las imágenes de salida en la carpeta: {OUTPUT_IMG_DIR}")


Generando imagen de muestra para la época 1...
Iniciando Época 1/10


Época 1: 100%|██████████| 1956/1956 [04:07<00:00,  7.90batch/s, D_loss: 0.9276, G_loss_total: 21.1970]



--- Resumen Época 1 ---
Tiempo: 252.53 seg
Pérdida Discriminador (Media): 0.9274
Pérdida Generador (Total):     21.2067
Pérdida Generador (GAN):       1.5370
Pérdida Generador (L1):        0.1967
---------------------------


Generando imagen de muestra para la época 2...
Iniciando Época 2/10


Época 2: 100%|██████████| 1956/1956 [04:01<00:00,  8.12batch/s, D_loss: 0.8499, G_loss_total: 20.8277]



--- Resumen Época 2 ---
Tiempo: 241.47 seg
Pérdida Discriminador (Media): 0.8497
Pérdida Generador (Total):     20.8252
Pérdida Generador (GAN):       1.6692
Pérdida Generador (L1):        0.1916
---------------------------


Generando imagen de muestra para la época 3...
Iniciando Época 3/10


...


--- Resumen Época 8 ---
Tiempo: 243.91 seg
Pérdida Discriminador (Media): 0.8323
Pérdida Generador (Total):     19.7746
Pérdida Generador (GAN):       1.7095
Pérdida Generador (L1):        0.1807
---------------------------


Generando imagen de muestra para la época 9...
Iniciando Época 9/10


Época 9: 100%|██████████| 1956/1956 [04:03<00:00,  8.04batch/s, D_loss: 0.8515, G_loss_total: 19.6644]



--- Resumen Época 9 ---
Tiempo: 243.72 seg
Pérdida Discriminador (Media): 0.8527
Pérdida Generador (Total):     19.6555
Pérdida Generador (GAN):       1.6779
Pérdida Generador (L1):        0.1798
---------------------------


Generando imagen de muestra para la época 10...
Iniciando Época 10/10


Época 10: 100%|██████████| 1956/1956 [04:01<00:00,  8.10batch/s, D_loss: 0.8382, G_loss_total: 19.6591]



--- Resumen Época 10 ---
Tiempo: 241.83 seg
Pérdida Discriminador (Media): 0.8371
Pérdida Generador (Total):     19.6622
Pérdida Generador (GAN):       1.7208
Pérdida Generador (L1):        0.1794
---------------------------

Checkpoint guardado para la época 10 en ./training_checkpoints
Generando imagen final del entrenamiento...
¡Entrenamiento completado!
Puedes ver las imágenes de salida en la carpeta: Resultados_Entrenamiento
